# Programación y rivales: ¿cuándo y contra quién juegan los grandes en Santiago?
**Entrega 03 · Visualización complementaria · Temporada 2024**

Este notebook procesa la base de partidos de local de Colo-Colo, Universidad de Chile y Universidad Católica (2024) y genera una visualización en Altair sobre **programación** (fecha y horario) y **tipo de rival** (clásico o regular).

Pasos:
1. Instalar librerías y cargar el CSV
2. Limpieza y variables nuevas
3. Revisión de los datos procesados
4. Visualización en Altair
5. Exportación (HTML y JPG)

## 1. Instalación y carga de datos

In [1]:
!pip install -q altair vl-convert-python pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.5/33.5 MB 26.5 MB/s eta 0:00:00


In [2]:
import pandas as pd
import altair as alt
from PIL import Image

print('Altair', alt.__version__)

Altair 5.5.0


Sube el archivo `partidos_asistencia_y_precios_2024_completo.csv` al panel de archivos de Colab (ícono de carpeta, a la izquierda) o ejecuta la celda siguiente para subirlo.

In [3]:
# Solo en Colab: sube el CSV desde tu computador
from google.colab import files
subido = files.upload()

Saving partidos_2024.csv to partidos_2024 (1).csv


In [5]:
# Carga del CSV original (el que se subió en la celda anterior)
RUTA_CSV = 'partidos_2024.csv'
df = pd.read_csv(RUTA_CSV)
print(f'Cargando archivo: {RUTA_CSV}')
print(df.shape)
df.head()

Cargando archivo: partidos_2024.csv
(45, 13)


,Equipo,Fecha N°,Fecha,Hora,Rival,Estadio,Goles Local,Goles Visita,Resultado,Aforo Autorizado,Público Controlado,% Ocupación,Precio Más Bajo (CLP)
0,Colo-Colo,Fecha 3,03/03/2024,18:00,Huachipato,Estadio Monumental,2,0,Victoria,32000,NaN,-,NaN
1,Colo-Colo,Fecha 4,10/03/2024,18:00,Universidad de Chile,Estadio Monumental,0,1,Derrota,32000,NaN,-,NaN
2,Colo-Colo,Fecha 6,30/03/2024,18:00,Everton,Estadio Monumental,4,1,Victoria,30000,NaN,-,NaN
3,Colo-Colo,Fecha 8,15/04/2024,20:30,Cobreloa,Estadio Monumental,0,2,Derrota,35000,NaN,-,NaN
4,Colo-Colo,Fecha 10,28/04/2024,18:00,Unión La Calera,Estadio Monumental,3,1,Victoria,30000,NaN,-,NaN


## 2. Limpieza y variables nuevas

**Decisiones de limpieza**
- `Fecha` se convierte a formato de fecha (formato original día/mes/año).
- `Hora` se convierte a número decimal (17:30 → 17,5) para poder ubicarla en un eje.
- Se crea el **día de la semana** a partir de la fecha.
- Se unifican los nombres de rivales: "Iquique" y "Deportes Iquique" son el mismo equipo.
- Se crea **Tipo de partido**: *Clásico* si el rival es otro de los tres equipos del estudio (Colo-Colo, Universidad de Chile, Universidad Católica); *Regular* en los demás casos.
- `% Ocupación` se convierte a número. Los vacíos ("-") quedan como `NaN`, sin imputar, para ser transparentes con los datos faltantes.

In [6]:
# 1. Fechas y horas
# La fecha viene como dd/mm/aaaa en el CSV original; si el archivo ya fue procesado, viene como aaaa-mm-dd.
# Se prueba primero el formato ISO y luego el formato día/mes/año, para no confundir día con mes.
fecha_iso = pd.to_datetime(df['Fecha'], format='%Y-%m-%d', errors='coerce')
fecha_dmy = pd.to_datetime(df['Fecha'], format='%d/%m/%Y', errors='coerce')
df['Fecha'] = fecha_iso.fillna(fecha_dmy)
assert df['Fecha'].notna().all(), 'Hay fechas que no se pudieron interpretar'
hora = pd.to_datetime(df['Hora'], format='%H:%M')
df['Hora decimal'] = hora.dt.hour + hora.dt.minute / 60

# 2. Día de la semana
dias = {0: 'Lunes', 1: 'Martes', 2: 'Miércoles', 3: 'Jueves',
        4: 'Viernes', 5: 'Sábado', 6: 'Domingo'}
df['Día de la semana'] = df['Fecha'].dt.dayofweek.map(dias)
df['Tipo de día'] = df['Fecha'].dt.dayofweek.map(
    lambda d: 'Fin de semana' if d >= 5 else 'Día de semana')

# 3. Nombres de rivales
df['Rival'] = df['Rival'].replace({'Iquique': 'Deportes Iquique'})

# 4. Tipo de partido
grandes = ['Colo-Colo', 'Universidad de Chile', 'Universidad Católica']
df['Tipo de partido'] = df['Rival'].apply(
    lambda r: 'Clásico' if r in grandes else 'Regular')

# 5. Franja horaria
def franja(h):
    if h < 17:
        return 'Tarde (antes de 17:00)'
    if h < 19:
        return 'Tarde-noche (17:00-18:59)'
    return 'Noche (19:00 o más)'
df['Franja horaria'] = df['Hora decimal'].apply(franja)

# 6. Ocupación como número (los vacíos quedan como NaN)
df['% Ocupación'] = pd.to_numeric(
    df['% Ocupación'].astype(str).str.replace('%', '').str.replace(',', '.').str.strip(),
    errors='coerce')

df = df.sort_values(['Equipo', 'Fecha']).reset_index(drop=True)
df.to_csv('partidos_2024_procesado.csv', index=False)
df.head()

,Equipo,Fecha N°,Fecha,Hora,Rival,Estadio,Goles Local,Goles Visita,Resultado,Aforo Autorizado,Público Controlado,% Ocupación,Precio Más Bajo (CLP),Hora decimal,Día de la semana,Tipo de día,Tipo de partido,Franja horaria
0,Colo-Colo,Fecha 3,2024-03-03,18:00,Huachipato,Estadio Monumental,2,0,Victoria,32000,NaN,NaN,NaN,18.0,Domingo,Fin de semana,Regular,Tarde-noche (17:00-18:59)
1,Colo-Colo,Fecha 4,2024-03-10,18:00,Universidad de Chile,Estadio Monumental,0,1,Derrota,32000,NaN,NaN,NaN,18.0,Domingo,Fin de semana,Clásico,Tarde-noche (17:00-18:59)
2,Colo-Colo,Fecha 6,2024-03-30,18:00,Everton,Estadio Monumental,4,1,Victoria,30000,NaN,NaN,NaN,18.0,Sábado,Fin de semana,Regular,Tarde-noche (17:00-18:59)
3,Colo-Colo,Fecha 8,2024-04-15,20:30,Cobreloa,Estadio Monumental,0,2,Derrota,35000,NaN,NaN,NaN,20.5,Lunes,Día de semana,Regular,Noche (19:00 o más)
4,Colo-Colo,Fecha 10,2024-04-28,18:00,Unión La Calera,Estadio Monumental,3,1,Victoria,30000,NaN,NaN,NaN,18.0,Domingo,Fin de semana,Regular,Tarde-noche (17:00-18:59)


## 3. Revisión de los datos procesados

In [7]:
print('Partidos por equipo y tipo de partido')
print(pd.crosstab(df['Equipo'], df['Tipo de partido']))
print()
print('Partidos por tipo de día')
print(df['Tipo de día'].value_counts())
print()
print('Partidos por franja horaria y equipo')
print(pd.crosstab(df['Equipo'], df['Franja horaria']))
print()
print('Aforo autorizado promedio por equipo y tipo de partido')
print(df.groupby(['Equipo', 'Tipo de partido'])['Aforo Autorizado'].mean().round(0))
print()
print('Datos faltantes de ocupación:', df['% Ocupación'].isna().sum(), 'de', len(df))

Partidos por equipo y tipo de partido
Tipo de partido       Clásico  Regular
Equipo                                
Colo-Colo                   2       13
Universidad Católica        2       13
Universidad de Chile        2       13

Partidos por tipo de día
Tipo de día
Fin de semana    37
Día de semana     8
Name: count, dtype: int64

Partidos por franja horaria y equipo
Franja horaria        Noche (19:00 o más)  Tarde (antes de 17:00)  \
Equipo                                                              
Colo-Colo                               3                       4   
Universidad Católica                    6                       1   
Universidad de Chile                    2                       3   

Franja horaria        Tarde-noche (17:00-18:59)  
Equipo                                           
Colo-Colo                                     8  
Universidad Católica                          8  
Universidad de Chile                         10  

Aforo autorizado promedio po

## 4. Visualización

Cada punto es un partido de local. El eje horizontal es la fecha, el vertical la hora de inicio, el tamaño el aforo autorizado y el color distingue clásicos de partidos regulares. Hay un panel por equipo.

In [8]:
COLOR_CLASICO = '#d62828'
COLOR_REGULAR = '#8d99ae'

# Etiquetas de hora en formato hh:mm para el eje vertical
etiqueta_hora = "datum.value + ':00'"

base = alt.Chart(df).mark_circle(opacity=0.85, stroke='white', strokeWidth=1).encode(
    x=alt.X('Fecha:T',
            title=None,
            axis=alt.Axis(format='%b', labelAngle=0, grid=False)),
    y=alt.Y('Hora decimal:Q',
            title='Hora de inicio',
            scale=alt.Scale(domain=[11.5, 21.5]),
            axis=alt.Axis(values=[12, 14, 16, 18, 20], labelExpr=etiqueta_hora)),
    size=alt.Size('Aforo Autorizado:Q',
                  title='Aforo autorizado',
                  scale=alt.Scale(range=[60, 700]),
                  legend=alt.Legend(format=',.0f', values=[10000, 20000, 30000, 40000])),
    color=alt.Color('Tipo de partido:N',
                    title='Tipo de partido',
                    scale=alt.Scale(domain=['Clásico', 'Regular'],
                                    range=[COLOR_CLASICO, COLOR_REGULAR])),
    tooltip=[
        alt.Tooltip('Equipo:N', title='Local'),
        alt.Tooltip('Rival:N'),
        alt.Tooltip('Fecha:T', title='Fecha', format='%d/%m/%Y'),
        alt.Tooltip('Día de la semana:N', title='Día'),
        alt.Tooltip('Hora:N', title='Hora'),
        alt.Tooltip('Estadio:N'),
        alt.Tooltip('Resultado:N'),
        alt.Tooltip('Aforo Autorizado:Q', title='Aforo autorizado', format=',.0f'),
    ],
).properties(width=620, height=150)

chart = (
    base.facet(row=alt.Row('Equipo:N', title=None,
                           header=alt.Header(labelAngle=0, labelAlign='left',
                                             labelFontSize=13, labelFontWeight='bold')))
    .properties(title=alt.TitleParams(
        text='¿Cuándo y contra quién juegan los grandes de Santiago?',
        subtitle=['Partidos de local de Colo-Colo, U. de Chile y U. Católica en 2024.',
                  'Los 6 clásicos se jugaron en fin de semana y ninguno de noche; los partidos nocturnos y de semana fueron regulares.'],
        anchor='start', fontSize=18, subtitleFontSize=12, subtitleColor='#555'))
    .configure(background='white')
    .configure_view(stroke=None)
    .configure_axis(labelFontSize=11, titleFontSize=12)
    .configure_legend(titleFontSize=12, labelFontSize=11)
)
chart

alt.FacetChart(...)

## 5. Exportación

In [9]:
# HTML interactivo (con tooltips)
chart.save('visualizacion_programacion_rivales.html')

# JPG: Altair exporta a PNG con vl-convert y luego se convierte a JPG
chart.save('visualizacion_programacion_rivales.png', scale_factor=2)
Image.open('visualizacion_programacion_rivales.png').convert('RGB').save(
    'visualizacion_programacion_rivales.jpg', quality=95)

print('Archivos generados:')
print('- partidos_2024_procesado.csv')
print('- visualizacion_programacion_rivales.html')
print('- visualizacion_programacion_rivales.jpg')

Archivos generados:
- partidos_2024_procesado.csv
- visualizacion_programacion_rivales.html
- visualizacion_programacion_rivales.jpg


In [10]:
# Descarga los archivos desde Colab
from google.colab import files
for f in ['partidos_2024_procesado.csv',
          'visualizacion_programacion_rivales.html',
          'visualizacion_programacion_rivales.jpg']:
    files.download(f)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>